In [ ]:
# 1. Фиксим конфликт версий на Kaggle
!pip uninstall torchao peft -y
!pip install transformers datasets evaluate rouge_score accelerate sentencepiece -q
!pip install peft==0.11.1 -q # <-- Фиксируем стабильную версию

# 2. Автоматическая перезагрузка ядра, чтобы изменения вступили в силу
import os
os._exit(0) 

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 7.8 MB/s eta 0:00:00


In [1]:
# 2. Импорт и проверка железа
import os
import json
import numpy as np
from pathlib import Path
import torch
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType # <-- ДОБАВЛЕНО

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram_gb:.1f} GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
print("Loading IlyaGusev/gazeta dataset...")
dataset = load_dataset("IlyaGusev/gazeta")

# УВЕЛИЧЕНО: 20000 для обучения, 1000 для валидации
train_dataset = dataset["train"].select(range(20000))  # <-- БЫЛО 5000
eval_dataset = dataset["test"].select(range(1000))     # <-- БЫЛО 500

print(f"Train size: {len(train_dataset)}")
print(f"Eval size: {len(eval_dataset)}")

Loading IlyaGusev/gazeta dataset...


README.md: 0.00B [00:00, ?B/s]

default/train/0000.parquet:   0%|          | 0.00/252M [00:00<?, ?B/s]

default/train/0001.parquet:   0%|          | 0.00/22.7M [00:00<?, ?B/s]

default/validation/0000.parquet:   0%|          | 0.00/27.8M [00:00<?, ?B/s]

default/test/0000.parquet:   0%|          | 0.00/30.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60964 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6369 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6793 [00:00<?, ? examples/s]

Train size: 20000
Eval size: 1000


In [3]:
# 4. Загрузка модели и токенизатора
MODEL_CHECKPOINT = "google/mt5-small"

print(f"Loading {MODEL_CHECKPOINT}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

print(f"Base model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

Loading google/mt5-small...


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Base model parameters: 556.3M


In [4]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,              # <-- УВЕЛИЧЕНО с 8 до 16
    lora_alpha=32,     # <-- УВЕЛИЧЕНО с 16 до 32
    lora_dropout=0.1,
    target_modules=["q", "v", "k", "o", "wi", "wo"],
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,769,472 || all params: 558,060,928 || trainable%: 0.3171


In [5]:
prefix = "summarize: "  # <-- ИЗМЕНЕНО с русского на английский
max_input_length = 512    
max_target_length = 128   

def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["text"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(text_target=examples["summary"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Preprocessing train dataset...")
tokenized_train = train_dataset.map(preprocess_function, batched=True)

print("Preprocessing eval dataset...")
tokenized_eval = eval_dataset.map(preprocess_function, batched=True)

Preprocessing train dataset...


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Preprocessing eval dataset...


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [6]:
# 7. Метрики (твой отличный код с защитой от -100)
import nltk
nltk.download("punkt", quiet=True)
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    predictions = predictions.astype(int)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]
    
    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {
        "rouge1": float(result["rouge1"]),
        "rouge2": float(result["rouge2"]),
        "rougeL": float(result["rougeL"])
    }

In [7]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-small-gazeta-lora",
    eval_strategy="epoch",
    save_strategy="epoch",           # <-- ДОБАВЬ ЭТУ СТРОКУ
    learning_rate=1e-4,
    per_device_train_batch_size=2,      
    per_device_eval_batch_size=4,       
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=4,
    predict_with_generate=True,
    fp16=True,                          
    gradient_checkpointing=True,        
    gradient_accumulation_steps=8,      
    logging_steps=50,
    report_to="none",
    generation_max_length=128,          
    generation_num_beams=4,
    warmup_steps=0.1,
    optim="adamw_torch",                
    dataloader_num_workers=0,
    load_best_model_at_end=True,
    metric_for_best_model="rouge1",
    greater_is_better=True,
)

In [8]:
# 9. Инициализация Trainer
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [9]:
# 10. Очистка памяти и запуск
import gc
gc.collect()
torch.cuda.empty_cache()

print("Starting training with LoRA...")
train_result = trainer.train()

metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

Starting training with LoRA...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
1,66.588345,2.648770,0.098964,0.027893,0.098303
2,62.809697,2.587703,0.165330,0.048502,0.162196
3,61.268643,2.557735,0.168139,0.049130,0.164710
4,60.788740,2.552640,0.172431,0.052046,0.168920


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


***** train metrics *****
  epoch                    =        4.0
  total_flos               = 39799921GF
  train_loss               =    82.7528
  train_runtime            = 5:06:25.57
  train_samples_per_second =      4.351
  train_steps_per_second   =      0.136


In [10]:
# 11. Сохранение модели
# ВАЖНО: При использовании LoRA мы сохраняем и базовую модель, и адаптеры вместе, 
# чтобы потом использовать её как обычную модель в Streamlit.
output_dir = "/kaggle/working/mt5-small-gazeta-finetuned-lora"
Path(output_dir).mkdir(exist_ok=True)

# merge_and_unload объединяет адаптеры с базовой моделью в один цельный файл
model_to_save = model.merge_and_unload() 
model_to_save.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Merged Model saved to {output_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Merged Model saved to /kaggle/working/mt5-small-gazeta-finetuned-lora


In [11]:
# 12. Финальная оценка (Быстрый инференс через Trainer)
print("Running final evaluation...")
predictions_output = trainer.predict(tokenized_eval)

preds = predictions_output.predictions
labels = predictions_output.label_ids

# Декодирование
preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
decoded_preds = tokenizer.batch_decode(preds.astype(int), skip_special_tokens=True)
labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

final_result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

print("\n" + "=" * 70)
print("=== FINAL METRICS (mT5-small + LoRA, 512 tokens) ===")
print("=" * 70)
print(f"ROUGE-1: {float(final_result['rouge1']):.4f}")
print(f"ROUGE-2: {float(final_result['rouge2']):.4f}")
print(f"ROUGE-L: {float(final_result['rougeL']):.4f}")
print("=" * 70)

# Сохранение
results = {
    'model': 'mt5-small-finetuned-lora-gazeta',
    'max_input_length': 512,
    'metrics': {
        'rouge1': float(final_result['rouge1']), 
        'rouge2': float(final_result['rouge2']), 
        'rougeL': float(final_result['rougeL'])
    }
}
with open('/kaggle/working/mt5_small_lora_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

Running final evaluation...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



=== FINAL METRICS (mT5-small + LoRA, 512 tokens) ===
ROUGE-1: 0.1724
ROUGE-2: 0.0520
ROUGE-L: 0.1689
